# Final Load & KPI Computation
This notebook computes high-level executive findings and prepares final summaries.

### Load Cleaned Data
Loading data from `data/processed/gtd_cleaned.csv`.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(''))) if '__file__' not in globals() else os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
input_path = os.path.join(BASE_DIR, 'data', 'processed', 'gtd_cleaned.csv')
outputs_dir = os.path.join(BASE_DIR, 'notebooks', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

df = pd.read_csv(input_path)

## 1. Key Performance Indicators (KPIs) Summary
We compute the aggregate global indicators from the cleaned data.

In [ ]:
total_incidents = len(df)
total_casualties = df['casualties'].sum()
top_country = df['country'].value_counts().idxmax()
top_region = df['region'].value_counts().idxmax()
common_attack_type = df['attack_type'].value_counts().idxmax()
yearly_counts = df['year'].value_counts()
peak_year = yearly_counts.idxmax()
avg_casualties_per_incident = df['casualties'].mean()

known_groups = df[df['group_name'].str.lower() != 'unknown']
top_3_groups = known_groups['group_name'].value_counts().head(3).index.tolist()

kpi_data = {
    "Metric": [
        "Total incidents in cleaned dataset",
        "Total casualties (killed + wounded)",
        "Most targeted country (by incident count)",
        "Most targeted region (by incident count)",
        "Most common attack type",
        "Peak year of terrorism",
        "Average casualties per incident",
        "Top 3 terrorist groups"
    ],
    "Value": [
        total_incidents,
        int(total_casualties),
        top_country,
        top_region,
        common_attack_type,
        peak_year,
        round(avg_casualties_per_incident, 2),
        ", ".join(top_3_groups)
    ]
}

kpi_df = pd.DataFrame(kpi_data)
kpi_csv_path = os.path.join(BASE_DIR, 'data', 'processed', 'kpi_summary.csv')
kpi_df.to_csv(kpi_csv_path, index=False)

print("KPI Summary Table:")
print(kpi_df.to_string(index=False))

### Executive Summary of KPIs in Business Language
- **Total Magnitude:** There have been a total of **181,691** incidents in the original GTD dataset, reduced to significant verified incidents in the cleaned dataset.
- **Geographic Impact:** The most highly targeted country is structurally impacted within the broader region, which itself leads all regions globally in incident volume.
- **Attack Method:** The most prominent stratagem for these incidents accounts for a significant plurality of attacks.
- **Temporal Peak:** Global activity saw its most intense peak heavily in the past decades.
- **Incident Severity:** On average, each incident results in measurable casualties highlighting the human cost.
- **Leading Perpetrators:** The top identifiable organizations driving these incidents represent key geopolitical concerns.

## 2. Regional Shift & Volatility Matrix
A pivot matrix showing how regional violence distribution shifted over subsequent decades. Exported to `data/processed/regional_volatility_matrix.csv`.

In [ ]:
def get_decade(year):
    return (year // 10) * 10

df['decade'] = df['year'].apply(get_decade)
matrix_df = pd.crosstab(df['region'], df['decade'])
matrix_df.columns = [f"{col}s" for col in matrix_df.columns]

output_matrix = os.path.join(BASE_DIR, 'data', 'processed', 'regional_volatility_matrix.csv')
matrix_df.to_csv(output_matrix)

print("Regional Shift & Volatility Matrix:")
print(matrix_df.head().to_string())

plt.figure(figsize=(12, 8))
sns.heatmap(matrix_df, cmap='Reds', annot=False)
plt.title('Heatmap of Regional Violance via Decade')
plt.ylabel('Region')
plt.xlabel('Decade')
plt.tight_layout()
plt.savefig(os.path.join(outputs_dir, 'regional_volatility_heatmap.png'))
plt.show()